# 📑 Automated Reviewer Assignment & COI Flow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/visvesh-sathe/reviewer_assignment/blob/main/coi_gen_and_reviewer_allocation/reviewer_assignment_colab.ipynb)


Welcome! This notebook provides an interactive, end-to-end pipeline to:
1. **Install dependencies & setup workspace**
2. **Download reference template files & upload custom inputs**
3. **Generate Conflict of Interest (COI) matrix & report**
4. **Download COI matrix for review/editing**
5. **Upload updated COI matrix**
6. **Run Reviewer Allocation optimization (MILP)**
7. **Produce visuals, charts & download deliverables package**

---
## 🛠️ Step 1: Install Dependencies & Setup Workspace
Installs `pulp` MILP solver and optimization libraries, sets up the workspace from GitHub or uploaded archive, and sets the working directory.

In [ ]:
#@title Step 1: Install Dependencies & Setup Workspace { display-mode: "form" }
import os
import sys
import shutil
from pathlib import Path

# 1. Install required packages
!pip install -q "pulp>=2.8,<4.0" pyyaml pandas matplotlib ipywidgets

# 2. Always ensure a fresh, complete copy from GitHub
repo_dir = Path("/content/reviewer_assignment")
expected_file = repo_dir / "coi_gen_and_reviewer_allocation" / "input_files" / "cfg_paper_info.csv"

if repo_dir.exists() and not expected_file.exists():
    print("⚠️ Found incomplete or stale repository directory. Cleaning up...")
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    print("🌐 Fetching latest repository from GitHub...")
    !git clone https://github.com/visvesh-sathe/reviewer_assignment.git /content/reviewer_assignment
else:
    print("🔄 Forcing sync to latest commit from GitHub...")
    !cd /content/reviewer_assignment && git fetch origin main && git reset --hard origin/main

print("\n📌 Active Git Commit in Colab:")
!cd /content/reviewer_assignment && git log -n 1 --oneline

# 3. Enter workspace directory
workspace = repo_dir / "coi_gen_and_reviewer_allocation"
if workspace.exists():
    os.chdir(str(workspace))
    print(f"\n✅ Environment Ready! Current directory: {os.getcwd()}")
    print("\n📁 Files available in input_files/:")
    for f in sorted(os.listdir("input_files")):
        print(f"   • {f}")
    print("\n📁 Files available in reference_templates/:")
    for f in sorted(os.listdir("reference_templates")):
        print(f"   • {f}")
else:
    print("⚠️ Could not locate workspace directory. Please re-run this cell.")


---
## 📤 Step 2: Reference Templates & Input Files Panel
In this panel, you can:
1. **Download Reference Files:** Download sample template CSV files (or the full ZIP) to see the expected columns and formatting.
2. **Upload Your Completed Files:** Upload your own updated files to replace any or all of the inputs.

💡 *If you do not upload a file, the pre-loaded reference file is used automatically!*

In [ ]:
#@title Step 2: Reference Templates & Input Files Panel { display-mode: "form" }
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files

# Auto-recover working directory if user skipped Step 1
workspace = Path("/content/reviewer_assignment/coi_gen_and_reviewer_allocation")
if not os.path.exists("input_files") and workspace.exists():
    os.chdir(str(workspace))

if not os.path.exists("input_files") or not os.path.exists("reference_templates"):
    print("⚠️ Workspace files not found. Please click the Play button on Step 1 first!")
else:
    # ----------------- SECTION A: DOWNLOAD REFERENCE TEMPLATES -----------------
    ref_dropdown = widgets.Dropdown(
        options=[
            ("📦 All 5 Reference Templates (ZIP Package)", "reference_templates/reference_templates.zip"),
            ("1. Reviewer Expertise (cfg_reviewer_expertise.csv)", "reference_templates/reference_reviewer_expertise.csv"),
            ("2. Paper Info / Submissions (cfg_paper_info.csv)", "reference_templates/reference_paper_info.csv"),
            ("3. Paper Technical Areas (cfg_paper_area.csv)", "reference_templates/reference_paper_area.csv"),
            ("4. Must-Review Constraints (cfg_must_review.csv)", "reference_templates/reference_must_review.csv"),
            ("5. Existing COI Matrix (cfg_coi.csv)", "reference_templates/reference_coi.csv"),
            ("6. Master Config (cfg_config.yaml)", "reference_templates/reference_config.yaml"),
            ("7. Deliverable Sheet Template (reference_assignment_sheet_template.xls)", "reference_templates/reference_assignment_sheet_template.xls")
        ],
        description="Template:",
        layout=widgets.Layout(width="440px")
    )

    btn_download_ref = widgets.Button(
        description="⬇️ Download Selected File",
        button_style="info",
        icon="download",
        layout=widgets.Layout(width="240px")
    )

    def on_download_ref_clicked(b):
        path = ref_dropdown.value
        if os.path.exists(path):
            print(f"⬇️ Downloading '{os.path.basename(path)}' to your browser...")
            files.download(path)
        else:
            print(f"Error: '{path}' not found.")

    btn_download_ref.on_click(on_download_ref_clicked)

    # ----------------- SECTION B: UPLOAD CUSTOM FILES -----------------
    btn_expertise = widgets.FileUpload(description="1. Expertise", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_info = widgets.FileUpload(description="2. Paper Info", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_areas = widgets.FileUpload(description="3. Tech Areas", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_must = widgets.FileUpload(description="4. Must-Review", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_coi = widgets.FileUpload(description="5. COI (Optional)", accept=".csv", multiple=False, layout=widgets.Layout(width="200px"))
    btn_config = widgets.FileUpload(description="6. Config (Optional)", accept=".yaml,.yml", multiple=False, layout=widgets.Layout(width="200px"))

    btn_apply = widgets.Button(description="💾 Save & Stage Uploaded Files", button_style="success", icon="check", layout=widgets.Layout(width="280px", height="38px"))
    output_status = widgets.Output()

    def save_widget_file(widget_obj, target_path):
        if not widget_obj.value:
            return False
        if isinstance(widget_obj.value, (tuple, list)):
            item = widget_obj.value[0]
            content = item.get("content")
            with open(target_path, "wb") as f:
                f.write(bytes(content))
            return True
        elif isinstance(widget_obj.value, dict):
            first_val = next(iter(widget_obj.value.values()))
            content = first_val.get("content")
            with open(target_path, "wb") as f:
                f.write(bytes(content))
            return True
        return False

    def on_apply_clicked(b):
        with output_status:
            output_status.clear_output()
            saved = []
            if save_widget_file(btn_expertise, "input_files/cfg_reviewer_expertise.csv"):
                saved.append("cfg_reviewer_expertise.csv")
            if save_widget_file(btn_info, "input_files/cfg_paper_info.csv"):
                saved.append("cfg_paper_info.csv")
            if save_widget_file(btn_areas, "input_files/cfg_paper_area.csv"):
                saved.append("cfg_paper_area.csv")
            if save_widget_file(btn_must, "input_files/cfg_must_review.csv"):
                saved.append("cfg_must_review.csv")
            if save_widget_file(btn_coi, "input_files/cfg_coi.csv"):
                saved.append("cfg_coi.csv")
            if save_widget_file(btn_config, "input_files/cfg_config.yaml"):
                saved.append("cfg_config.yaml")
            
            if saved:
                print("✅ Successfully staged custom uploaded files:")
                for s in saved:
                    print(f"   • input_files/{s}")
            else:
                print("ℹ️ No new files selected. Pre-loaded reference files in input_files/ are active.")
            
            print("\n📁 Current files active in input_files/:")
            for f in sorted(os.listdir("input_files")):
                size_kb = os.path.getsize(os.path.join("input_files", f)) / 1024
                print(f"   • {f:<28} ({size_kb:.1f} KB)")

    btn_apply.on_click(on_apply_clicked)

    # Render UI
    display(HTML("<h3>Part A: Download Reference Templates</h3>"))
    display(HTML("<p>Select any template to download as an example format, or download all 5 as a zip:</p>"))
    display(widgets.HBox([ref_dropdown, btn_download_ref]))

    display(HTML("<br><hr><h3>Part B: Upload Your Files</h3>"))
    display(HTML("<p>Select your completed files below, then click <b>'Save & Stage Uploaded Files'</b>:</p>"))
    display(widgets.VBox([
        widgets.HBox([widgets.Label("1. Reviewer Expertise:", layout=widgets.Layout(width="180px")), btn_expertise]),
        widgets.HBox([widgets.Label("2. Paper Info / Submissions:", layout=widgets.Layout(width="180px")), btn_info]),
        widgets.HBox([widgets.Label("3. Paper Technical Areas:", layout=widgets.Layout(width="180px")), btn_areas]),
        widgets.HBox([widgets.Label("4. Must-Review Constraints:", layout=widgets.Layout(width="180px")), btn_must]),
        widgets.HBox([widgets.Label("5. Existing COI (Optional):", layout=widgets.Layout(width="180px")), btn_coi]),
        widgets.HBox([widgets.Label("6. Master Config (Optional):", layout=widgets.Layout(width="180px")), btn_config]),
        widgets.HTML("<br>"),
        btn_apply,
        output_status
    ]))


---
## 🔍 Step 3: Run COI Generation
Extracts author affiliations from `cfg_paper_info.csv` and matches them against reviewer affiliations from `cfg_reviewer_expertise.csv` to generate the Conflict of Interest (COI) matrix and audit report.

In [ ]:
#@title Step 3: Generate COI Matrix & Audit Report { display-mode: "form" }
import os

!python3 coi_generation.py

report_path = "output_files/coi_report.txt"
if os.path.exists(report_path):
    print("\n" + "="*50)
    print("📋 CONFLICT OF INTEREST REPORT PREVIEW:")
    print("="*50)
    with open(report_path, "r") as f:
        print(f.read())


---
## ⬇️ Step 4: Download COI Matrix for Review
Downloads the newly generated `cfg_coi.csv`. You can open this CSV file in Excel or Google Sheets to inspect the detected conflicts and make manual additions or removals (1 for conflict, 0 for clear).

In [ ]:
#@title Step 4: Download Generated COI Matrix { display-mode: "form" }
from google.colab import files
import os

coi_file = "input_files/cfg_coi.csv"
if os.path.exists(coi_file):
    print(f"⬇️ Downloading '{coi_file}' to your browser...")
    files.download(coi_file)
else:
    print(f"Error: '{coi_file}' not found. Please run Step 3 first.")


---
## ⬆️ Step 5: Upload Modified COI Matrix
Upload your revised `cfg_coi.csv`. This will overwrite `input_files/cfg_coi.csv` and be used in the optimization solver in Step 6.
*(If you made no manual changes, you can simply skip this step and proceed to Step 6)*.

In [ ]:
#@title Step 5: Upload Modified COI Matrix (Optional) { display-mode: "form" }
from google.colab import files
import os

print("📤 Select your edited 'cfg_coi.csv' file to upload (or skip if no edits):")
uploaded = files.upload()

if uploaded:
    for fname in uploaded.keys():
        os.replace(fname, "input_files/cfg_coi.csv")
        print(f"✅ Successfully staged '{fname}' as 'input_files/cfg_coi.csv'.")
else:
    print("ℹ️ No file uploaded. Continuing with autogenerated COI matrix.")


---
## 🚀 Step 6: Run Reviewer Allocation (MILP Optimizer)
Runs the Mixed-Integer Linear Programming optimization to assign reviewers, appoint balanced Champions and Co-Champions (excluding Chair/Co-Chair), and maximize matching expertise.

**Workload Balancing:**
- Reviews per paper: loaded from `cfg_config.yaml` (default: 4), adjustable below in the form.
- Minimum reviews per reviewer: $\lfloor (\text{num\_papers} \times \text{reviews\_per\_paper}) / \text{num\_reviewers} \rfloor$
- Maximum reviews per reviewer: $\text{min} + \text{max\_extra\_reviews}$

In [ ]:
#@title Step 6: Execute Reviewer Allocation Flow { display-mode: "form" }
#@markdown **Reviewer Workload Settings:**
#@markdown Number of reviews required per paper (default loaded from `cfg_config.yaml`):
reviews_per_paper = 4 #@param {type:"integer"}

#@markdown Adjust `max_extra_reviews` (default: 2) to control how many reviews above the minimum fair baseline any reviewer can take:
max_extra_reviews = 2 #@param {type:"integer"}

import os
import yaml

# Read defaults from cfg_config.yaml if available
config_file = "input_files/cfg_config.yaml"
yaml_reviews_per_paper = 4
yaml_max_extra = 2
if os.path.exists(config_file):
    try:
        with open(config_file, "r") as f:
            cfg_data = yaml.safe_load(f) or {}
            yaml_reviews_per_paper = int(cfg_data.get("reviews_per_paper", cfg_data.get("min_reviews_per_paper", 4)))
            yaml_max_extra = int(cfg_data.get("max_extra_reviews", 2))
    except Exception as e:
        pass

# Fallback to YAML default if non-positive value entered
active_reviews_per_paper = reviews_per_paper if reviews_per_paper > 0 else yaml_reviews_per_paper
active_max_extra = max_extra_reviews if max_extra_reviews >= 0 else yaml_max_extra

print(f"🚀 Running Reviewer Allocation Optimizer:")
print(f"   • reviews_per_paper: {active_reviews_per_paper} (YAML default: {yaml_reviews_per_paper})")
print(f"   • max_extra_reviews: {active_max_extra} (YAML default: {yaml_max_extra})\n")

!python3 reviewer_allocation.py --reviews-per-paper {active_reviews_per_paper} --max-extra-reviews {active_max_extra}

---
## 📊 Step 7: View Visualizations & Download Deliverables
Displays all workload distribution charts inline and downloads the final  package containing all assignment matrices, vectors, summary reports, and the filled deliverable spreadsheet .

In [ ]:
#@title Step 7: Display Charts & Download Complete Deliverables Package { display-mode: "form" }
import glob
import os
from IPython.display import Image, display
from google.colab import files

print("📊 Workload & Assignment Distributions:\n")
chart_images = sorted(glob.glob("output_files/*.png"))
if chart_images:
    for img_path in chart_images:
        print(f"\n📈 {os.path.basename(img_path)}")
        display(Image(img_path, width=700))
else:
    print("No charts found in output_files/.")

if os.path.exists("output_files/final_reviewer_assignment_sheet.xls"):
    print("📄 Found deliverable spreadsheet: output_files/final_reviewer_assignment_sheet.xls (included in download package)")

!zip -r -q /content/reviewer_assignment_results.zip output_files/
print("\n⬇️ Downloading deliverables package: reviewer_assignment_results.zip")
files.download("/content/reviewer_assignment_results.zip")
